In [1]:
from pathlib import Path
path_spec_data=Path.cwd().parent.parent/"spec_data"
path_benchmark_data=Path.cwd().parent.parent/"benchmark_search_global_repository"

path_spec_data.mkdir(parents=True, exist_ok=True)
path_benchmark_data.mkdir(parents=True, exist_ok=True)

In [ ]:
# Only record hybrid search

original_library_size = ['repository']
    
query_size = 100

ion_mode = [-1, 1,]

steps=[
    "open_search", 
    "neutral_loss_search", 
    "hybrid_search"]



dynamic_script_path="21_dynamic_entropy_search_repository_search.py"


In [3]:
import subprocess
import pickle
import os
import time
import shutil
from typing import Union
import numpy as np
from dynamic_entropy_search.dynamic_entropy_search import DynamicEntropySearch
from dynamic_entropy_search.convert_to_mgf import _write_spec


def run_usrbintime_by_arguments(
          arguments:list[str], 
          if_output:bool=False, 
          output_memory_file:Union[str,Path]=None, 
          output_time_file:Union[str, Path]=None):
    
    # arguments: script_path, str(charge), step
    command=["/usr/bin/time","-v","python"] + arguments

    if if_output: # Output to files as record
        with open(output_memory_file, "w") as f1, open(output_time_file, "w") as f2:
            subprocess.run(command, stderr=f1, stdout=f2, cwd=Path.cwd(), env=os.environ.copy())

    else: # Output is not needed
         
        subprocess.run(command, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL, cwd=Path.cwd(), env=os.environ.copy())
        
    return

def perform_search_and_record_one_by_one(
          script_path:Path, 
          charge:int,
          library_size:int,
          step:str,
          query_spectra_list:list,
          method:str,
          if_output:bool=True, 
          ):
    
    for i, spec in enumerate(query_spectra_list):
        # Generate temp query file to perform search one by one
        temp_query_spec=path_spec_data/f"benchmark_spec/query_spectra-charge_{charge}_temp.pkl"
        with open(temp_query_spec, "wb") as temp1:

            # Write temp query pickle file
            pickle.dump(spec, temp1)

        
        arguments=[script_path, str(charge), temp_query_spec, step]
        output_memory_file=path_benchmark_data/f"{method}_{charge}_{library_size}_memory_usage_{step}_step_query_{i}.txt"
        output_time_file=path_benchmark_data/f"{method}_{charge}_{library_size}_compare_time_{step}_step_query_{i}.txt"
        
        # Perform search and record memory usage and time
        run_usrbintime_by_arguments(arguments=arguments, if_output=if_output, output_memory_file=output_memory_file, output_time_file=output_time_file)
    

    return


for library_size in original_library_size:

    for charge in ion_mode:
        for step in steps:
            
            if step=="open_search" or step=="neutral_loss_search" or step=="hybrid_search":

                query_pkl=path_spec_data/f"benchmark_spec/query_spectra-charge_{charge}-number_100.pkl"
                query_spectra=pickle.loads(open(query_pkl, "rb").read())

                ### DynamicEntropySearch ###
                method="repository"
                    
                perform_search_and_record_one_by_one(
                    script_path=dynamic_script_path,
                    charge=charge,
                    library_size=library_size,
                    step=step,
                    query_spectra_list=query_spectra,
                    method=method,
                    if_output=True
                )
                    

                
                            
